App Deleter for Windows 

GhostOkaami : guzmanwolfrank @Github

In [2]:
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import subprocess
import winreg
import os
import threading
from typing import List, Dict

class ProgramUninstaller:
    def __init__(self, root):
        self.root = root
        self.root.title("Program Uninstaller")
        self.root.geometry("600x500")
        self.root.resizable(True, True)
        
        # Store installed programs
        self.installed_programs = []
        
        self.setup_ui()
        self.load_installed_programs()
    
    def setup_ui(self):
        # Main frame
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Configure grid weights
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(1, weight=1)
        main_frame.rowconfigure(2, weight=1)
        
        # Title
        title_label = ttk.Label(main_frame, text="Program Uninstaller", 
                               font=('Arial', 16, 'bold'))
        title_label.grid(row=0, column=0, columnspan=3, pady=(0, 20))
        
        # Search section
        ttk.Label(main_frame, text="Search Program:").grid(row=1, column=0, 
                                                           sticky=tk.W, pady=5)
        
        self.search_var = tk.StringVar()
        self.search_var.trace('w', self.filter_programs)
        search_entry = ttk.Entry(main_frame, textvariable=self.search_var, width=40)
        search_entry.grid(row=1, column=1, sticky=(tk.W, tk.E), pady=5, padx=(5, 0))
        
        refresh_btn = ttk.Button(main_frame, text="Refresh List", 
                                command=self.load_installed_programs)
        refresh_btn.grid(row=1, column=2, pady=5, padx=(5, 0))
        
        # Program list
        ttk.Label(main_frame, text="Installed Programs:").grid(row=2, column=0, 
                                                              columnspan=3, sticky=tk.W, pady=(10, 5))
        
        # Treeview with scrollbar
        tree_frame = ttk.Frame(main_frame)
        tree_frame.grid(row=3, column=0, columnspan=3, sticky=(tk.W, tk.E, tk.N, tk.S), pady=5)
        tree_frame.columnconfigure(0, weight=1)
        tree_frame.rowconfigure(0, weight=1)
        
        self.program_tree = ttk.Treeview(tree_frame, columns=('Publisher', 'Version'), 
                                        show='tree headings', height=12)
        self.program_tree.heading('#0', text='Program Name', anchor=tk.W)
        self.program_tree.heading('Publisher', text='Publisher', anchor=tk.W)
        self.program_tree.heading('Version', text='Version', anchor=tk.W)
        
        # Configure column widths
        self.program_tree.column('#0', width=300, minwidth=200)
        self.program_tree.column('Publisher', width=150, minwidth=100)
        self.program_tree.column('Version', width=100, minwidth=80)
        
        scrollbar = ttk.Scrollbar(tree_frame, orient=tk.VERTICAL, 
                                 command=self.program_tree.yview)
        self.program_tree.configure(yscrollcommand=scrollbar.set)
        
        self.program_tree.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        scrollbar.grid(row=0, column=1, sticky=(tk.N, tk.S))
        
        # Buttons frame
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=4, column=0, columnspan=3, pady=(10, 0))
        
        self.uninstall_btn = ttk.Button(button_frame, text="Uninstall Selected", 
                                       command=self.uninstall_program, 
                                       style='Accent.TButton')
        self.uninstall_btn.pack(side=tk.LEFT, padx=(0, 10))
        
        info_btn = ttk.Button(button_frame, text="Program Info", 
                             command=self.show_program_info)
        info_btn.pack(side=tk.LEFT)
        
        # Status bar
        self.status_var = tk.StringVar()
        self.status_var.set("Ready")
        status_bar = ttk.Label(main_frame, textvariable=self.status_var, 
                              relief=tk.SUNKEN, anchor=tk.W)
        status_bar.grid(row=5, column=0, columnspan=3, sticky=(tk.W, tk.E), 
                       pady=(10, 0))
        
        # Progress bar (initially hidden)
        self.progress = ttk.Progressbar(main_frame, mode='indeterminate')
        self.progress.grid(row=6, column=0, columnspan=3, sticky=(tk.W, tk.E), 
                          pady=(5, 0))
        self.progress.grid_remove()  # Hide initially
    
    def load_installed_programs(self):
        """Load installed programs from Windows registry"""
        self.status_var.set("Loading installed programs...")
        self.progress.grid()
        self.progress.start()
        
        def load_in_thread():
            self.installed_programs = []
            
            # Registry paths to check
            reg_paths = [
                (winreg.HKEY_LOCAL_MACHINE, 
                 r"SOFTWARE\Microsoft\Windows\CurrentVersion\Uninstall"),
                (winreg.HKEY_LOCAL_MACHINE, 
                 r"SOFTWARE\WOW6432Node\Microsoft\Windows\CurrentVersion\Uninstall"),
                (winreg.HKEY_CURRENT_USER, 
                 r"SOFTWARE\Microsoft\Windows\CurrentVersion\Uninstall")
            ]
            
            for hkey, reg_path in reg_paths:
                try:
                    with winreg.OpenKey(hkey, reg_path) as key:
                        for i in range(winreg.QueryInfoKey(key)[0]):
                            try:
                                subkey_name = winreg.EnumKey(key, i)
                                with winreg.OpenKey(key, subkey_name) as subkey:
                                    program_info = self.get_program_info(subkey)
                                    if program_info and program_info['display_name']:
                                        self.installed_programs.append(program_info)
                            except Exception:
                                continue
                except Exception:
                    continue
            
            # Sort programs by name
            self.installed_programs.sort(key=lambda x: x['display_name'].lower())
            
            # Update UI in main thread
            self.root.after(0, self.update_program_list)
        
        # Start loading in background thread
        thread = threading.Thread(target=load_in_thread, daemon=True)
        thread.start()
    
    def get_program_info(self, subkey) -> Dict:
        """Extract program information from registry key"""
        try:
            display_name = winreg.QueryValueEx(subkey, "DisplayName")[0]
            
            # Skip system components and updates
            if any(skip in display_name.lower() for skip in 
                   ['microsoft visual c++', 'microsoft .net', 'update for', 
                    'security update', 'hotfix']):
                return None
            
            info = {'display_name': display_name}
            
            # Get optional fields
            try:
                info['publisher'] = winreg.QueryValueEx(subkey, "Publisher")[0]
            except FileNotFoundError:
                info['publisher'] = "Unknown"
            
            try:
                info['version'] = winreg.QueryValueEx(subkey, "DisplayVersion")[0]
            except FileNotFoundError:
                info['version'] = "Unknown"
            
            try:
                info['uninstall_string'] = winreg.QueryValueEx(subkey, "UninstallString")[0]
            except FileNotFoundError:
                info['uninstall_string'] = ""
            
            try:
                info['install_location'] = winreg.QueryValueEx(subkey, "InstallLocation")[0]
            except FileNotFoundError:
                info['install_location'] = ""
            
            return info
            
        except Exception:
            return None
    
    def update_program_list(self):
        """Update the program list in the UI"""
        self.progress.stop()
        self.progress.grid_remove()
        
        # Clear existing items
        for item in self.program_tree.get_children():
            self.program_tree.delete(item)
        
        # Add programs to tree
        for program in self.installed_programs:
            self.program_tree.insert('', 'end', 
                                   text=program['display_name'],
                                   values=(program['publisher'], program['version']))
        
        self.status_var.set(f"Loaded {len(self.installed_programs)} programs")
    
    def filter_programs(self, *args):
        """Filter programs based on search text"""
        search_text = self.search_var.get().lower()
        
        # Clear existing items
        for item in self.program_tree.get_children():
            self.program_tree.delete(item)
        
        # Add filtered programs
        filtered_count = 0
        for program in self.installed_programs:
            if (search_text in program['display_name'].lower() or 
                search_text in program['publisher'].lower()):
                self.program_tree.insert('', 'end', 
                                       text=program['display_name'],
                                       values=(program['publisher'], program['version']))
                filtered_count += 1
        
        if search_text:
            self.status_var.set(f"Found {filtered_count} matching programs")
        else:
            self.status_var.set(f"Showing all {len(self.installed_programs)} programs")
    
    def get_selected_program(self):
        """Get the currently selected program"""
        selection = self.program_tree.selection()
        if not selection:
            return None
        
        item = self.program_tree.item(selection[0])
        program_name = item['text']
        
        # Find the program in our list
        for program in self.installed_programs:
            if program['display_name'] == program_name:
                return program
        return None
    
    def show_program_info(self):
        """Show detailed information about selected program"""
        program = self.get_selected_program()
        if not program:
            messagebox.showwarning("No Selection", "Please select a program first.")
            return
        
        info_window = tk.Toplevel(self.root)
        info_window.title(f"Program Information - {program['display_name']}")
        info_window.geometry("500x300")
        info_window.resizable(True, True)
        
        # Create scrolled text widget
        text_widget = scrolledtext.ScrolledText(info_window, wrap=tk.WORD, 
                                               width=60, height=15)
        text_widget.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Display program information
        info_text = f"""Program Name: {program['display_name']}
Publisher: {program['publisher']}
Version: {program['version']}
Install Location: {program['install_location']}
Uninstall Command: {program['uninstall_string']}"""
        
        text_widget.insert(tk.END, info_text)
        text_widget.config(state=tk.DISABLED)
    
    def uninstall_program(self):
        """Uninstall the selected program"""
        program = self.get_selected_program()
        if not program:
            messagebox.showwarning("No Selection", "Please select a program to uninstall.")
            return
        
        # Confirm uninstallation
        response = messagebox.askyesno(
            "Confirm Uninstall",
            f"Are you sure you want to uninstall '{program['display_name']}'?\n\n"
            "This action cannot be undone."
        )
        
        if not response:
            return
        
        if not program['uninstall_string']:
            messagebox.showerror("Error", 
                               "No uninstall command found for this program.")
            return
        
        try:
            self.status_var.set(f"Uninstalling {program['display_name']}...")
            
            # Run the uninstall command
            uninstall_cmd = program['uninstall_string']
            
            # Add silent flags if it's an MSI installer
            if 'msiexec' in uninstall_cmd.lower():
                if '/quiet' not in uninstall_cmd.lower():
                    uninstall_cmd += ' /quiet'
            
            subprocess.run(uninstall_cmd, shell=True, check=False)
            
            messagebox.showinfo("Uninstall Started", 
                              f"Uninstall process for '{program['display_name']}' has been initiated.\n"
                              "Please follow any additional prompts that may appear.")
            
            self.status_var.set("Uninstall process started")
            
            # Offer to refresh the list
            if messagebox.askyesno("Refresh List", 
                                 "Would you like to refresh the program list?"):
                self.load_installed_programs()
                
        except Exception as e:
            messagebox.showerror("Error", f"Failed to uninstall program:\n{str(e)}")
            self.status_var.set("Uninstall failed")

def main():
    # Check if running on Windows
    if os.name != 'nt':
        print("This program only works on Windows.")
        return
    
    root = tk.Tk()
    app = ProgramUninstaller(root)
    root.mainloop()

if __name__ == "__main__":
    main()